# TDMEC Embedding Pilot (Kaggle GPU)

**Model name:** TDMEC (Temporal Dynamic Multiplex Evolutionary Community)

This notebook runs the bounded Qwen3 embedding pilot (default 10k+10k), Stage-B pooling,
graph–text alignment, and `TDMEC_INPUT` export.

Labels: `PROVISIONAL_SMOKE_ONLY` / `ENGINEERING_VALIDATION` / `NOT_FOR_FINAL_THESIS_CONCLUSIONS`

Requires authenticated Hugging Face Hub access via `HF_TOKEN` (Kaggle Secret).
Restartable: re-run from the top after setting dataset paths. No Google Drive dependency.


In [ ]:
# 1) Install dependencies (Kaggle GPU session)
import os, sys, subprocess
from pathlib import Path

CODE_ROOT = Path('/kaggle/working/community-evolution-modeling')
CANDIDATES = [
    Path('/kaggle/input/tdmec-embedding-code'),
    Path('/kaggle/input/tdmec-embedding-code-20260804'),
    Path('/kaggle/working'),
]
print('Looking for code package…')
for c in CANDIDATES:
    print(' ', c, 'exists' if c.exists() else 'missing')


In [ ]:
# Unpack transfer tarball if present, else assume repo already extracted
import tarfile

WORKING = Path('/kaggle/working')
REPO = WORKING / 'community-evolution-modeling'
tarballs = list(Path('/kaggle/input').glob('**/tdmec_embedding_code_transfer_*.tar.gz'))
if tarballs and not (REPO / 'src' / 'tdmec_embeddings').exists():
    print('Extracting', tarballs[0])
    with tarfile.open(tarballs[0], 'r:gz') as tf:
        tf.extractall(WORKING)
print('REPO', REPO, 'exists=', REPO.exists())
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

req = REPO / 'requirements' / 'embeddings-target-studio.txt'
if req.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
else:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'accelerate', 'safetensors', 'psutil', 'pyyaml', 'pyarrow', 'numpy',
    ])


In [ ]:
# 2) Resolve Kaggle dataset mounts (private) + HF auth
def find_run(root_name_parts, run_id):
    for p in Path('/kaggle/input').rglob('manifest.json'):
        try:
            import json
            m = json.loads(p.read_text())
            if m.get('run_id') == run_id:
                return p.parent
        except Exception:
            pass
    raise FileNotFoundError(run_id)

DATASET_A = find_run(['smoke', 'a'], 'smoke_a_pg_001')
DATASET_B = find_run(['smoke', 'b'], 'smoke_b_pg_001')
OUT = Path('/kaggle/working/tdmec_embeddings')
OUT.mkdir(parents=True, exist_ok=True)

os.environ['TDMEC_DATASET_A_ROOT'] = str(DATASET_A)
os.environ['TDMEC_DATASET_B_ROOT'] = str(DATASET_B)
os.environ['TDMEC_EMBEDDING_OUTPUT_ROOT'] = str(OUT)
# Immutable pin from successful preflight (also hardcoded in pilot YAML)
REV = '5cf2132abc99cad020ac570b19d031efec650f2b'
os.environ['QWEN3_MODEL_REVISION'] = REV
os.environ['QWEN3_TOKENIZER_REVISION'] = REV

# Authenticated Hugging Face Hub access (required for model download)
hf_token = (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN') or '').strip()
assert hf_token, (
    'Set HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) in Kaggle Secrets / env. '
    'Anonymous Hub download is not supported by the TDMEC encoder.'
)
print('A', DATASET_A)
print('B', DATASET_B)
print('OUT', OUT)
print('HF_TOKEN set:', True)
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch probe skipped:', exc)


In [ ]:
# 3) Dry-run source verification (smoke config)
from tdmec_embeddings.run_pilot_embedding import main as pilot_main
rc = pilot_main([
    '--config', 'configs/qwen3_tdmec_smoke_64.yaml',
    '--dry-run',
])
assert rc == 0, rc


In [ ]:
# 4) Smoke first: 64 nodes + 64 events
# Dual auth for pilot later; smoke needs --authorize-real-model only.
# --replace-incomplete clears a previous failed attempt (never COMPLETED).
rc = pilot_main([
    '--config', 'configs/qwen3_tdmec_smoke_64.yaml',
    '--authorize-real-model',
    '--replace-incomplete',
])
assert rc == 0, rc

# 5) After smoke review: bounded 10k pilot (uncomment to run)
# rc = pilot_main([
#     '--config', 'configs/qwen3_tdmec_pilot.yaml',
#     '--authorize-real-model',
#     '--authorize-bounded-pilot',
#     '--replace-incomplete',
# ])
# assert rc == 0, rc


In [ ]:
# 6) Final package validation report
import json
from pathlib import Path
pkgs = sorted(Path('/kaggle/working/tdmec_embeddings').glob('TDMEC_INPUT_*'))
print('packages', pkgs)
assert pkgs, 'TDMEC_INPUT package not found'
pkg = pkgs[-1]
from tdmec_embeddings.validation import validate_tdmec_input_package
report = validate_tdmec_input_package(pkg, expected_dimension=512)
print(json.dumps(report, indent=2)[:4000])
assert report['passed'], report.get('failures')
print('READY:', pkg)
